In [0]:
import os
import sys
import numpy as np
import rasterio
import geopandas as gpd
import pandas as pd
from skimage.filters import threshold_otsu
from rasterstats import zonal_stats
from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings
from functools import reduce
from operator import mul
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling
from shapely.geometry import mapping
import re
from rasterio.mask import mask
from rasterio.io import MemoryFile
import tools
from pathlib import Path
import pipelines
import mosaicVI
import traceback

In [0]:

CATALOG = "use1_prod_artemis_catalog_3718194974443840"
SCHEMA = "tier1_raw" 

TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_mission_table"
df_misiones = spark.table(TABLE_NAME)

flights = df_misiones.collect()

# Variable para el total de vuelos
total_vuelos = len(flights)
print(f"Total flights recorded in the table: {total_vuelos}")

processed = 0
skipped = 0
errors = 0

# Usamos enumerate nativo de Python
for i, vuelo in enumerate(flights, 1):
    
    # Imprimimos el progreso en texto plano
    print(f"\n[{i}/{total_vuelos}] Reviewing DEM inventory for mission: {vuelo['mission']} | Lote: {vuelo['field']}...")
    
       # Rutas dinámicas
    fd = vuelo["flight_metadata_path"]
    date_folder = os.path.dirname(fd) 
    mission_folder = os.path.dirname(date_folder)
    
    field_data_dir = os.path.join(mission_folder, "field_data")
    polygon_file_path = f"{field_data_dir}/plot_boundary.geojson"

    # Guardamos el CSV en la carpeta de la misión para no ensuciar los rasters
    output_csv_DEM = f"{mission_folder}/ouput_DEM.csv"
    
    # Validar si el CSV ya existe en el Volumen
    if not os.path.exists(output_csv_DEM):
        try:
            # 💡 INSTANCIACIÓN LIMPIA: Eliminamos sys.setprofile(None) que causaba el conflicto
            results_DEM = pipelines.DEM()
            
            # Ejecutar tu código de procesamiento DEM pasándole 'mission_folder'
            results_DEM.analyze_dem(
                mission_folder, 
                polygon_file_path, 
                output_csv_DEM
            )
            print(f" DEM analysis completed and CSV generated in: {output_csv_DEM}")
            processed += 1
            
        except Exception as e:
            print(f" Error processing the DEM for the mission {vuelo['mission']}: {e}")
            # Esto imprimirá la traza exacta si llega a fallar algo internamente
            traceback.print_exc()
            errors += 1
            
    else:
        print(f" ⏭ Skipped. CSV already exists.")
        skipped += 1

# Reporte final
print("\n" + "="*45)
print("BATCH DEM PROCESSING RESULTS")
print("="*45)
print(f"▶ New cases: {processed}")
print(f"▶ Skipped (already existed): {skipped}")
print(f"▶ Execution errors: {errors}")

Total flights recorded in the table: 7

[1/7] Reviewing DEM inventory for mission: mission_idk | Lote: bhavanisagar...
 ⏭ Skipped. CSV already exists.

[2/7] Reviewing DEM inventory for mission: mission_100 | Lote: field_100...
 ⏭ Skipped. CSV already exists.

[3/7] Reviewing DEM inventory for mission: prueba_3 | Lote: field_06...
 ⏭ Skipped. CSV already exists.

[4/7] Reviewing DEM inventory for mission: corn_health_assessment_flight | Lote: west_field_03...
 ⏭ Skipped. CSV already exists.

[5/7] Reviewing DEM inventory for mission: prueba_2 | Lote: west_field_03...
 ⏭ Skipped. CSV already exists.

[6/7] Reviewing DEM inventory for mission: prueba_4 | Lote: field_10...
 ⏭ Skipped. CSV already exists.

[7/7] Reviewing DEM inventory for mission: prueba_5 | Lote: field_11...
 ⏭ Skipped. CSV already exists.

BATCH DEM PROCESSING RESULTS
▶ New cases: 0
▶ Skipped (already existed): 7
▶ Execution errors: 0


In [0]:

CATALOG = "use1_prod_artemis_catalog_3718194974443840"
SCHEMA = "tier1_raw" 

TABLE_NAME = f"{CATALOG}.{SCHEMA}.drone_mission_table"
df_misiones = spark.table(TABLE_NAME)
flights = df_misiones.collect()

# Variable para el total de vuelos
total_vuelos = len(flights)

processed = 0
skipped = 0
errors = 0

# Usamos enumerate nativo de Python
for i, vuelo in enumerate(flights, 1):
    
    # Imprimimos el progreso en texto plano
    print(f"\n[{i}/{total_vuelos}] Checking inventory for mission: {vuelo['mission']} | Lote: {vuelo['field']}...")
    
    # Rutas dinámicas
    fd = vuelo["flight_metadata_path"]
    date_folder = os.path.dirname(fd) 
    mission_folder = os.path.dirname(date_folder)
    
    field_data_dir = os.path.join(mission_folder, "field_data")
    polygon_file_path = f"{field_data_dir}/plot_boundary.geojson"
    
    # Guardamos el CSV en la carpeta de la misión
    output_csv = f"{mission_folder}/ouput_INDEX.csv"
    
    if not os.path.exists(output_csv):
        try:
            raw_data_dir = os.path.join(date_folder, "raw_data")
            multi_spec_dir = os.path.join(raw_data_dir, "multi-spec")

            if os.path.exists(multi_spec_dir):
                
                results = pipelines.fourband() 
            
                results.process_multispectral_data(
                    mission_folder, 
                    polygon_file_path, 
                    output_csv,
                    r"",  
                    tools  # Asegúrate de tener 'tools' definido en tu notebook
                )
            else: 

                results = pipelines.RGB() 
                
                results.process_RGB_data(
                    mission_folder, 
                    polygon_file_path, 
                    output_csv,
                    r"",  
                    tools  # Asegúrate de tener 'tools' definido en tu notebook
                )

            print(f" CSV generated successfully in: {output_csv}")
            processed += 1
            
        except Exception as e:
            print(f" Error processing mission {vuelo['mission']}: {e}")
            # Esto imprimirá la traza exacta si llega a fallar algo internamente
            traceback.print_exc() 
            errors += 1
            
    else:
        print(f" Skipped. CSV already exists.")
        skipped += 1

# Reporte final
print("\n" + "="*45)
print("BATCH RGB PROCESSING RESULTS")
print("="*45)
print(f"▶ New cases: {processed}")
print(f"▶ Skipped (already existed): {skipped}")
print(f"▶ Execution errors: {errors}")


[1/7] Checking inventory for mission: mission_idk | Lote: bhavanisagar...
Processing VIS for folder: 2025-02-25
Shapefile loaded and reprojected
Calculating Otsu
THE ORIGINAL THRESHOLD IS 0.4331617057323456
Processing canopy metrics for folder: 2025-02-25
CRS: EPSG:4326  |  Pixel: 0.0094 m × 0.0095 m  |  Area: 0.000089 m²
Combined results exported successfully to /Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/bhavanisagar/best/2025:ind:soybean:unknown/bhavanisagar/section_a/drone/mission_idk/ouput_INDEX.csv
 CSV generated successfully in: /Volumes/use1_prod_artemis_catalog_3718194974443840/production/data/pheno_google/CIAT_CALI/bhavanisagar/best/2025:ind:soybean:unknown/bhavanisagar/section_a/drone/mission_idk/ouput_INDEX.csv

[2/7] Checking inventory for mission: mission_100 | Lote: field_100...
 Skipped. CSV already exists.

[3/7] Checking inventory for mission: prueba_3 | Lote: field_06...
 Skipped. CSV already exists.

[4/7] Checking inv